# 🧠 Sesión 8 · Cuando la red memoriza en vez de aprender

Imagina un alumno que se **aprende de memoria** las respuestas del examen del año pasado.
- Con **ese** examen saca un 10.
- Con un examen **nuevo**... suspende. ❌

A las redes les pasa lo mismo. Se llama **sobreajuste** (en inglés *overfitting*).

**Hoy veremos:** cómo detectarlo y cómo evitarlo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras

## 1. Pocos datos con "ruido"
Tenemos solo **20 puntos**. Siguen una curva suave, pero con algo de ruido (como los datos reales).

In [ ]:
np.random.seed(0)
X = np.linspace(-3, 3, 20)
y = X ** 2 + np.random.normal(0, 1.5, 20)

# Separamos: unos para aprender y otros para el examen
X_train, y_train = X[::2], y[::2]      # los pares
X_test,  y_test  = X[1::2], y[1::2]    # los impares

plt.scatter(X_train, y_train, label="Para aprender")
plt.scatter(X_test, y_test, label="Para el examen")
plt.legend()
plt.show()

## 2. Una red demasiado grande, entrenada demasiado tiempo
128 neuronas por capa y **2000 repeticiones** para solo 10 puntos.

💡 `validation_data` hace que, en cada repetición, la red también haga el **examen**. Así vemos las dos notas.

In [ ]:
grande = keras.Sequential([
    keras.Input(shape=(1,)),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(1)
])
grande.compile(loss="mean_squared_error", optimizer=keras.optimizers.Adam(0.01))

historial = grande.fit(X_train, y_train, epochs=2000, verbose=0,
                       validation_data=(X_test, y_test))
print("Entrenada ✅")

## 3. Las dos notas
- 🔵 **Error al aprender** (datos que ya conoce)
- 🟠 **Error en el examen** (datos nuevos)

In [ ]:
plt.plot(historial.history["loss"], label="Error al aprender")
plt.plot(historial.history["val_loss"], label="Error en el examen")
plt.yscale("log")
plt.xlabel("Repeticiones")
plt.legend()
plt.show()

👀 El error al aprender **baja y baja**, pero el del examen **deja de bajar o sube**.

🚨 **Esa es la señal de que la red está memorizando.**

Miramos la curva que ha aprendido:

In [ ]:
x_curva = np.linspace(-3, 3, 200)
plt.scatter(X_train, y_train, label="Para aprender")
plt.scatter(X_test, y_test, label="Para el examen")
plt.plot(x_curva, grande.predict(x_curva, verbose=0), color="red", label="Red")
plt.legend()
plt.show()

👀 La línea roja hace **curvas raras** para pasar por todos los puntos. Ha **memorizado** en vez de aprender la forma general.

## 4. La solución: parar a tiempo ⏱️
`EarlyStopping` vigila el error del examen. Si deja de mejorar, **para el entrenamiento** y se queda con la mejor versión.

In [ ]:
parar_a_tiempo = keras.callbacks.EarlyStopping(
    monitor="val_loss",            # vigila el error del examen
    patience=50,                   # espera 50 repeticiones antes de parar
    restore_best_weights=True      # se queda con la mejor versión
)

lista = keras.Sequential([
    keras.Input(shape=(1,)),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(1)
])
lista.compile(loss="mean_squared_error", optimizer=keras.optimizers.Adam(0.01))

historial2 = lista.fit(X_train, y_train, epochs=2000, verbose=0,
                       validation_data=(X_test, y_test),
                       callbacks=[parar_a_tiempo])
print("Ha parado en la repetición", len(historial2.history["loss"]))

## 5. Comparamos las dos redes

In [ ]:
plt.scatter(X_train, y_train, label="Para aprender")
plt.scatter(X_test, y_test, label="Para el examen")
plt.plot(x_curva, grande.predict(x_curva, verbose=0), color="red", label="Memoriza")
plt.plot(x_curva, lista.predict(x_curva, verbose=0), color="green", label="Para a tiempo")
plt.legend()
plt.show()

print("Error en el examen (memoriza):     ", round(grande.evaluate(X_test, y_test, verbose=0), 2))
print("Error en el examen (para a tiempo):", round(lista.evaluate(X_test, y_test, verbose=0), 2))

## 📝 Resumen
- **Sobreajuste** = la red **memoriza** los datos de entrenamiento y falla con datos nuevos.
- Se detecta cuando el **error del examen** (`val_loss`) deja de bajar.
- Soluciones:
  - **Parar a tiempo** → `EarlyStopping`
  - Usar una red **más pequeña**
  - Conseguir **más datos**